In [10]:
import torch
import numpy as np, cv2, pandas as pd, time
import matplotlib.pyplot as plt
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models


device = "cuda" if torch.cuda.is_available() else "cpu"
# print(f"Using device: {device}")

In [11]:
trn_df = pd.read_csv("Multi_Task_Learning/fairface-labels-train.csv")
val_df = pd.read_csv("Multi_Task_Learning/fairface-labels-val.csv")
print(trn_df.head())

          file  age  gender        race  service_test
0  train/1.jpg   59    Male  East Asian          True
1  train/2.jpg   39  Female      Indian         False
2  train/3.jpg   11  Female       Black         False
3  train/4.jpg   26  Female      Indian          True
4  train/5.jpg   26  Female      Indian          True


In [12]:
image_size = 224
class HumanDataset(Dataset):
    def __init__(self, df, trm=None):
        self.df = df
        self.trm = trm
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, index):
        x = self.df.iloc[index]
        file = "Multi_Task_Learning/" + x.file
        gender = x.gender == "Female"
        age = x.age
        # print(file)
        im = cv2.imread(file)
        im = cv2.cvtColor(im, cv2.COLOR_BGR2RGB)
        return im, age, gender
    
    def preprocess(self, im):
        im = cv2.resize(im, (image_size, image_size))
        im = torch.tensor(im).float() / 255.0
        im = im.permute(2,0,1)
        return im[None]
    
    def collate_fn(self, batch):
        ims, ages, genders = [], [], []
        for im, age, gender in batch:
            im = self.preprocess(im)
            ims.append(im)
            ages.append(float(int(age)/80))
            genders.append(float(gender))
            
        ages, genders = [torch.tensor(x).to(device).float() for x in [ages, genders]]
        ims = torch.cat(ims).to(device)
        
        return ims, ages, genders
    
        

In [13]:
train_dataset = HumanDataset(trn_df)
val_dataset = HumanDataset(val_df)

# print(len(train_dataset), len(val_dataset))
trn_dl = DataLoader(train_dataset, batch_size=32, shuffle=True, drop_last=True, collate_fn=train_dataset.collate_fn)
val_dl = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=val_dataset.collate_fn)

a, b, c = next(iter(trn_dl))
print(a.shape, b.shape, c.shape)


torch.Size([32, 3, 224, 224]) torch.Size([32]) torch.Size([32])


In [14]:
model = models.vgg19(pretrained=True)
for param in model.parameters():
    param.requires_grad = False
    
model.avgpool = nn.Sequential(
    nn.Conv2d(512, 512, kernel_size=3),
    nn.MaxPool2d(2),
    nn.ReLU(),
    nn.Flatten()
)

d:\Anaconda\envs\exercise\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
d:\Anaconda\envs\exercise\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG19_Weights.IMAGENET1K_V1`. You can also use `weights=VGG19_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [15]:

class HumanClassifier(nn.Module):
    def __init__(self):
        super(HumanClassifier, self).__init__()
        self.intermediate = nn.Sequential(
            nn.Linear(2048, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, 64),
            nn.ReLU(),
        )

        self.age_classifier = nn.Sequential(nn.Linear(64, 1), nn.Sigmoid())
        self.gender_classifier = nn.Sequential(nn.Linear(64, 1), nn.Sigmoid())
    
    def forward(self, x):
        x = self.intermediate(x)
        age = self.age_classifier(x)
        gender = self.gender_classifier(x)
        return age, gender
    
        

In [16]:
def train(model, train_loader, val_loader, criterion, optimizer, epochs=10):
    print("Starting training...")
    
    for epoch in range(epochs):
        train_loss = 0.0
        model.train()
        for ix, (im, age, gender) in enumerate(train_loader):
            optimizer.zero_grad()
            pred_gender, pred_age = model(im)
            gender_criterion, age_criterion = criterion
            # print(pred_gender.shape, gender.shape)
            gender_loss, age_loss = gender_criterion(pred_gender.squeeze(), gender), age_criterion(pred_age.squeeze(), age)
            total_loss = gender_loss + age_loss
            total_loss.backward()
            optimizer.step()
            train_loss += total_loss.item()
        avg_train_loss = train_loss / len(train_loader)
        
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for ix, (im, age, gender) in enumerate(val_loader):
                pred_gender, pred_age = model(im)
                gender_criterion, age_criterion = criterion
                gender_loss, age_loss = gender_criterion(pred_gender, gender), age_criterion(pred_age, age)
                total_loss = gender_loss + age_loss
                val_loss += total_loss.item()
                pred_gender = (pred_gender > 0.5).sequeeze()
        avg_val_loss = val_loss / len(val_loader)
        print(f"Epoch {epoch+1}/{epochs}, Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")
     
    return avg_train_loss, avg_val_loss
        
            

In [17]:
train_losses, val_losses = [], []
model.classifier = HumanClassifier()
gender_criterion = nn.BCELoss()
age_criterion = nn.L1Loss()
loss_functions = gender_criterion, age_criterion
model = model.to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-4)

val_gender_accs, val_age_maes = [], []
losses = train(model, trn_dl, val_dl, loss_functions, optimizer, epochs=10)

with torch.no_grad():
    val_gender_acc = 0.0
    val_age_mae = 0.0
    for ix, (im, age, gender) in enumerate(val_dl):
        pred_gender, pred_age = model(im)
        gender_loss, age_loss = gender_criterion(pred_gender, gender), age_criterion(pred_age, age)
        pred_gender = (pred_gender > 0.5).sequeeze()
        gender_acc = (pred_gender == gender).float().sum()
        age_mae = torch.abs(pred_age - age).float().sum()
        val_gender_acc += gender_acc.item()
        val_age_mae += age_mae.item()
    avg_val_gender_acc = val_gender_acc / len(val_dl)
    avg_val_age_mae = val_age_mae / len(val_dl)
val_gender_accs.append(avg_val_gender_acc)
val_age_maes.append(avg_val_age_mae)
epochs = list(range(1, 11))
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(epochs, val_gender_acc)
plt.subplot(1, 2, 2)
plt.plot(epochs, val_age_maes)
plt.show()



Starting training...


KeyboardInterrupt: 